# Graphen 2 

[Video5](https://youtu.be/9qFOS0HWmY4) - [Video6](https://youtu.be/28rTSSyJDQI)

### Kürzeste Wege

Die Länge eines Pfades in einem ungewichteten Graphen ist die Anzahl seiner Kanten. Die Entfernung zweier Knoten 
in einem ungewichteten Graphen ist die Länge des kürzesten Pfades zwischen ihnen.

<img src='graph11.png' width='350'>

<img src='graph50.png' width='351'>

Die Entfernung zwischen 'b' und 'i' ist 3.



Die **Breitensuche (breadth first search, bfs)** traversiert den Graphen in Schichten nach der Entfernung zum Ausgangsknoten.
Zu jedem Knoten merkt man sich seinen Vorgänger (prev), dadurch entsteht ein **shortest-path Baum**.
(Ein Baum ist ein zusammenhängender Graph, der keine Kreise enthält). Daraus lässt sich der kürzeste
Weg vom Ausgangsknoten zu jedem anderen Knoten rekonstruieren.

Der shortest-path Baum für die Breitensuche ab Knoten 'b':

<img src='graph11_sp_baum.png' width='350'>

    # Breitensuche mit Ausgangsknoten s:
    Für jeden Knoten u in G: 
        dist(u) = unendlich 
    prev = dict für den Vorgänger
    
    Füge s in eine Schlange Q ein 
    dist(s) = 0
    prev(s) = None
    
    Solange Q nicht leer:
        hole Knoten u aus der Schlange
            für alle Nachbarn v von u:
                falls dist(v) unendlich:
                    Füge v in die Schlange Q ein
                    dist(v) = dist(u) + 1
                    prev(v) = u

In [ ]:
from collections import deque

G = {
'a': set('bd'),
'b': set('acde'),
'c': set('bf'),
'd': set('abg'),
'e': set('bf'),
'f': set('cei'),
'g': set('dh'),
'h': set('fgi'),
'i': set('fh')
}

def reconstructPath(ziel,prev):
    path = []
    v = ziel
    while v is not None:
        path.append(v)
        v = prev[v]
    path.reverse()
    return path

inf = float('inf')
dist = {v:inf for v in G}
prev = dict()

s = 'b'
dist[s] = 0
prev[s] = None
Q = deque([s])          
while Q:
    u = Q.popleft()
    for v in G[u]:       
        if dist[v] == inf:
            Q.append(v)
            dist[v] = dist[u]+1
            prev[v] = u
            
ziel = 'i'
print(f'Entfernung von {s} nach {ziel} ist {dist[ziel]} mit Pfad {reconstructPath(ziel,prev)}')


### Der Algorithmus von Dijkstra

Der Algorithmus von Dijkstra findet  in einem gerichteten, mit nichtnegativen Kosten gewichteten
Graphen die kürzesten Wege von einem Startknoten zu allen anderen Knoten. Er löst das
**single source shortest path**-Problem.

<img src='graphen_26.png'>

    # Algorithmus von Dijkstra
    Setze dist des Startknotens s vorläufig auf 0,
        alle anderen vorläufig auf unendlich. 
    Setze prev des Startknotens auf None
    Solange es noch vorläufige Knoten gibt
        setze dist des billigsten vorläufigen Knotens u auf endgültig
        Für jeden Nachbarn v von u:   
            Falls (dist(u) + Kosten von u nach v) < dist(v):
                dist(v) =  dist(u) + Kosten von u nach v
                Setze Vorgänger von v auf u

In [1]:
G = {
'a': {'b':2, 'c':9},
'b': {'c':5, 'd':13},
'c': {'d':6, 'e':9},
'd': {'e':1, 'f':5},
'e': {'f':2},
'f': {}
}

from heapq import heapify, heappop, heappush

def reconstructPath(ziel,prev):
    path = []
    v = ziel
    while v is not None:
        path.append(v)
        v = prev[v]
    path.reverse()
    return path

    
inf = float('inf')
dist = {v:inf for v in G}
prev = dict()

s = 'a'         # startknoten
dist[s] = 0
prev[s] = None
endgueltig = set()    

vorlaeufig = [(dist[v],v) for v in G]
heapify(vorlaeufig)

while vorlaeufig:
    dist_u, u = heappop(vorlaeufig)
    if u in endgueltig: continue
    endgueltig.add(u)
    for v in G[u]:          
        if dist[v] > dist_u + G[u][v]:   # Relaxieren der Kante (u,v)
            dist[v] = dist_u + G[u][v]
            prev[v] = u
            heappush(vorlaeufig,(dist[v],v))
ziel = 'f'
print('Pfad von',s,'nach',ziel,':',*reconstructPath('f',prev))
print('Distanz:',dist[ziel])

Pfad von a nach f : a b c d e f
Distanz: 16


Während des Algorithmus müssen wir uns aus den vorläufig markierten Knoten laufend einen billigsten suchen. 
Dazu nutzen wir einen Heap. 
Die Laufzeit hängt ab vom Aufwand der Operationen zur Entnahme der Knoten aus dem Heap und dem Einfügen neuer Elemente in den Heap.
Bei einem Min-Heap geht beides in logarithmischer Zeit. Jeden Knoten müssen wir einmal aus dem Heap entfernen. Jede Kante kann dafür sorgen, dass neue Elemente in den Heap kommen. Insgesamt ist der Aufwand: $O((\left|V\right|+\left|E\right|)\cdot\log\left|V\right|)$. <br>
Mit einem Fibonacci-Heap kann dies auf $O(\left|V\right|+\left|E\right|)$ verbessert werden.
 

#### Relaxieren

Die Kante (u,v) *relaxieren* bedeutet: Prüfen, ob man über diese Kante besser zum Ziel v kommt und ggf. updaten.

### Der Algorithmus von Bellman-Ford

Der Algorithmus von **Bellman-Ford** kann auch mit negativen Kantengewichten umgehen, vorausgesetzt
es gibt keine Kreise mit negativem Gewicht.

Bei jedem Durchgang werden alle Kanten relaxiert. Beim ersten Durchgang hat man alle kürzesten Wege, die nur aus einer Kante bestehen, beim zweiten Durchgang hat man alle kürzesten Wege, die aus $<= 2$ Kanten bestehen usw. Da ein kürzester Pfad höchstens V-1 Kanten haben kann, ist der Algorithmus nach spätestens V-1 Durchgängen am Ziel. 



    # Algorithmus von Bellman-Ford
    Setze dist des Startknotens auf 0,
        alle anderen auf unendlich. 
    Setze prev des Startknotens auf None
    Solange sich was ändert:      # höchstens |V| - 1 mal
        Fur alle Kanten (u,v):
            Relaxiere(u,v)        # d.h. ggf. Kosten von v via u-v verbessern

Laufzeit: $O(\left|V\right|\cdot\left|E\right|)$.

<img src='graphen_27.png'>



    # Durchgänge bis sich nichts mehr ändert
    0 :  a:0 b:inf c:inf d:inf e:inf
    1 :  a:0 b:6 c:4 d:2 e:7
    2 :  a:0 b:2 c:4 d:2 e:7
    3 :  a:0 b:2 c:4 d:-2 e:7
    4 :  a:0 b:2 c:4 d:-2 e:7

In [3]:
G = {
'a': {'b':6,'e':7},
'b': {'d':-4,'e':8,'c':5},
'c': {'b':-2},
'd': {'c':7},
'e': {'c':-3,'d':9}
}

def reconstructPath(ziel,prev):
    path = []
    v = ziel
    while v is not None:
        path.append(v)
        v = prev[v]
    path.reverse()
    return path

inf = float('inf')
dist = {v:inf for v in G}
prev = dict()

start, ziel = 'a' , 'd'
dist[start] = 0
prev[start] = None

changed = True
while changed:
    changed = False
    for u in G:
        for v in G[u]:
            if dist[v] > dist[u] + G[u][v]:
                dist[v] = dist[u] + G[u][v]
                prev[v] = u
                changed = True

print('Pfad von',start,'nach',ziel,':',*reconstructPath(ziel,prev))
print('Distanz:',dist[ziel])

Pfad von a nach d : a e c b d
Distanz: -2


### Minimale Spannbäume

* Ein Teilgraph H eines ungerichteten Graphen G heisst **Spannbaum** von G, wenn H ein Baum auf den Knoten von
G ist. 

2. Ein Spannbaum S eines gewichteten, ungerichteten Graphen heisst **minimaler Spannbaum, (minimal spanning tree, MST)**, wenn S minimales Gewicht unter allen Spannbäumen von G besitzt.



#### Beispiele:
Man möchte Computer kostengünstig vernetzen.

<img src='graphen_28.png'>
<img src='graphen_29.png'>

Man möchte Orte mit möglichst kurzen Straßen verbinden.

<img src='graphen_30.png'> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;    <img src='graphen_31.png'>

 

#### Eigenschaften von Bäumen

* Ein Baum ist ein zusammenhängender, ungerichteter Graph, der keine Kreise enhält.
* Ein Baum mit $n$ Knoten hat $n-1$ Kanten.
* Jeder zusammenhängende, ungerichtete Graph mit $\left|V\right| = \left|E\right| - 1$ ist ein Baum.
* Ein ungerichteter Graph ist genau dann ein Baum, wenn es zwischen je zwei Knoten einen eindeutigen Pfad gibt.

### Der Algorithmus von Kruskal

Füge immer wieder die leichteste Kante hinzu, vorausgesetzt es entsteht kein Kreis.

<img src='graphen_32.png'>
<img src='graphen_33.png'>

a-d (5), c-e (5), d-f (6), a-b (7), b-e (7), e-g (9), Gesamtkosten:  39

Um zu verhindern, dass ein Kreis entsteht, wird jedem Knoten ein Repräsentant ('Chef') zugeordnet.
Eine Kante wird nur in den MST aufgenommen, wenn die beteilgten Knoten nicht denselben Chef haben.

In [5]:
'''
Algorithmus von Kruskal - einfache Version
'''
G = {
  'a': {'b':7, 'd':5},
  'b': {'a':7, 'd':9, 'e':7, 'c':8},
  'c': {'b':8, 'e':5},
  'd': {'a':5, 'b':9, 'e':15, 'f':6},
  'e': {'b':7, 'c':5, 'd':15, 'f':8, 'g':9},
  'f': {'d':6, 'e':8, 'g':11},
  'g': {'e':9, 'f':11}
}

def printMst(mst):
    for u,v in mst:
        print(f'{u}-{v} ({G[u][v]})', end=' ')

def find(u):                           # finde chef
    while chef[u] != u:                # chef zeigt auf sich selbst
        u = chef[u]
    return u

def union(u, v):
    u = find(u)                        # finde beide chefs
    v = find(v)
    chef[u] = v                        # der eine zeigt auf den anderen

summe = 0
gesehen = set()
E = []
for u in G:
    for v in G[u]:
        if (u,v) not in gesehen:
            if u < v:
                E.append((G[u][v],u,v))
            else:
                E.append((G[v][u],v,u))
            gesehen.add((u,v))
            gesehen.add((v,u))


# Kantenliste mit Kosten
E = sorted(E)                                   # nach Kosten sortieren
mst = []                                       
chef = {u:u for u in G}                         # zunächst ist jeder eigener chef
for _, u, v in E:
    if find(u) != find(v):                      # wenn chefs verschieden,
        mst.append((u, v))
        summe += G[u][v]
        union(u, v)
    if len(mst) == len(G) - 1:                  # von einem Startknoten aus gewinnen wir mit jeder
        break                                   # Kante einen neuen Knoten.
       

print(f'Minimaler Spannbaum, Gesamtkosten: {summe}') 
printMst(mst)


Minimaler Spannbaum, Gesamtkosten: 39
a-d (5) c-e (5) d-f (6) a-b (7) b-e (7) e-g (9) 

Das Ablaufprotokoll und der Chefbaum für die einfache Version des Kruskal-Algorithmus:

    Kantenliste: [(7, 'a', 'b'), (5, 'a', 'd'), (9, 'b', 'd'), (7, 'b', 'e'), (8, 'b', 'c'), (5, 'c', 'e'), (15, 'd', 'e'), (6, 'd', 'f'), (8, 'e', 'f'), (9, 'e', 'g'), (11, 'f', 'g')]
    
    a-d: 5 -  Chef von a wird d
    c-e: 5 -  Chef von c wird e
    d-f: 6 -  Chef von d wird f
    a-b: 7 -  Chef von f wird b
    b-e: 7 -  Chef von b wird e
    e-g: 9 -  Chef von e wird g
    
    Gesamtkosten: 39
    

<img src='graphen_34.png'>

Um lange Pfade im Chefbaum zu verhindern, wird der Algorithmus mit **union by rank** und **path compression** optimiert.

Wird der Chef von a gesucht, dann werden alle Zwischenknoten auf dem Weg zum Chef direkt mit
diesem verbunden. Jeder Knoten erhält einen Rang. Bei der Neuzuweisung eines
Chefs wird der Knoten mit dem höheren Rang Chef. Bei Gleichheit wird einer gewählt, dessen Rang
dann erhöht wird.

    a-d: 5 -  Rang a gleich Rang d: Chef von a wird d
    Rang von d wird 1
    c-e: 5 -  Rang c gleich Rang e: Chef von c wird e
    Rang von e wird 1
    d-f: 6 -  Rang d größer Rang f: Chef von f wird d
    a-b: 7 -  Rang d größer Rang b: Chef von b wird d
    b-e: 7 -  Rang d gleich Rang e: Chef von d wird e
    Rang von e wird 2
    Kompression: Chef von b wird e
    Kompression: Chef von f wird e
    e-g: 9 -  Rang e größer Rang g: Chef von g wird e
    a-d (5) c-e (5) d-f (6) a-b (7) b-e (7) e-g (9) Gesamtkosten:  39
    
    Chefbaum:
    {'a': 'd', 'b': 'e', 'c': 'e', 'd': 'e', 'e': 'e', 'f': 'e', 'g': 'e'}

<img src='graphen_35.png'>

In [3]:
G = {
  'a': {'b':7, 'd':5},
  'b': {'a':7, 'd':9, 'e':7, 'c':8},
  'c': {'b':8, 'e':5},
  'd': {'a':5, 'b':9, 'e':15, 'f':6},
  'e': {'b':7, 'c':5, 'd':15, 'f':8, 'g':9},
  'f': {'d':6, 'e':8, 'g':11},
  'g': {'e':9, 'f':11}
}

def printMst(mst):
    for u,v in mst:
        print(f'{u}-{v} ({G[u][v]})', end=' ')
        
def find(u):
    root = u                         # finde chef
    while chef[root] != root:        # chef zeigt auf sich selbst
        root = chef[root]
    while chef[u] != root:           # Path compression
        parent = chef[u]
        chef[u] = root
        u = parent
    return root

def union(u, v):
    u = find(u)                        # finde beide chefs
    v = find(v)
    if rank[u] > rank[v]:
        chef[v] = u
    else:
        chef[u] = v
    if rank[u] == rank[v]:             # bei gleichem rang wird v chef
        rank[v]+=1

summe = 0
gesehen = set()
E = []
for u in G:
    for v in G[u]:
        if (u,v) not in gesehen:
            if u < v:
                E.append((G[u][v],u,v))
            else:
                E.append((G[v][u],v,u))
            gesehen.add((u,v))
            gesehen.add((v,u))

# Kantenliste mit Kosten
E = sorted(E)                                   # nach Kosten sortieren
mst = []                                        # leere Teillösung
chef = {u:u for u in G}                         # zunächst ist jeder eigener chef
rank = {u:0 for u in G}
for _, u, v in E:
    if find(u) != find(v):                      # wenn chefs verschieden,
        mst.append((u, v))
        summe += G[u][v]
        union(u, v)
    if len(mst) == len(G) - 1:
        break


print(f'Minimaler Spannbaum, Gesamtkosten: {summe}') 
printMst(mst)

Minimaler Spannbaum, Gesamtkosten: 39
a-d (5) c-e (5) d-f (6) a-b (7) b-e (7) e-g (9) 

### Der Algorithmus von Jarnik-Prim

Gehe von einem Knoten aus und füge immer wieder einen neuen Knoten entlang der leichtesten Kante hinzu.

In [6]:
from heapq import heappop, heappush

G = {
  'a': {'b':7, 'd':5},
  'b': {'a':7, 'd':9, 'e':7, 'c':8},
  'c': {'b':8, 'e':5},
  'd': {'a':5, 'b':9, 'e':15, 'f':6},
  'e': {'b':7, 'c':5, 'd':15, 'f':8, 'g':9},
  'f': {'d':6, 'e':8, 'g':11},
  'g': {'e':9, 'f':11}
}

start = 'a'                              # Startknoten
mst = {}                                 # leere Teillösung
heap = [(0, None, start)]                # Heap mit Kanten und Kosten
summe = 0
while heap:
    cost, prev, u = heappop(heap)
    if u in mst: continue            # Zielknoten schon im Baum?
    mst[u] = prev                    # Kante von u nach prev kommt in mst
    summe += cost
    for v, cost in G[u].items():
        heappush(heap, (cost, u, v))

print(mst)
print('Gesamtkosten: ', summe)

{'a': None, 'd': 'a', 'f': 'd', 'b': 'a', 'e': 'b', 'c': 'e', 'g': 'e'}
Gesamtkosten:  39


Laufzeit Kruskal: Der wesentliche Aufwand ist das Sortieren der Kanten: $O(E \log(E))$. Union-Find ist durch Path Compression fast konstant schnell und daher vernachlässigbar.

Laufzeit Prim: Wir betrachten alle Kanten E. Bei einer Kante kann es notwendig sein, dass der Heap upgedated werden muss (im worst case bei allen Kanten): $O(E \log(V))$.

Faustregel: Kruskal eher bei dünn besetzten Graphen, Jarnik-Prim eher bei dicht besetzten Graphen.